# 02 - Nettoyage des données et prétraitement

## Introduction

Dans ce deuxième notebook du projet **Employee Attrition Prediction - RH**, l'objectif est de nettoyer les données et de préparer le dataset avant l'entraînement des modèles de classification.

Cette étape permet de supprimer les colonnes peu utiles, de séparer la variable cible des variables explicatives, d'identifier les variables numériques et catégorielles, puis de préparer les transformations nécessaires pour les futurs modèles.

## 1. Importation des bibliothèques

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

**Interprétation :** Ces bibliothèques sont utilisées pour manipuler les données, séparer le dataset en train/test et préparer les transformations nécessaires pour les modèles de classification.

## 2. Chargement du dataset

In [2]:
df = pd.read_csv("../data/raw/WA_Fn-UseC_-HR-Employee-Attrition.csv")
df.head()

,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,...,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,...,1,80,0,8,0,1,6,4,0,5
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,2,...,4,80,1,10,3,3,10,7,1,7
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,...,2,80,0,7,3,3,0,0,0,0
3,33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,1,5,...,3,80,0,8,3,3,8,7,3,0
4,27,No,Travel_Rarely,591,Research & Development,2,1,Medical,1,7,...,4,80,1,6,3,3,2,2,2,2


In [3]:
df.shape

(1470, 35)

**Interprétation :** Nous rechargeons le dataset afin de commencer la préparation des données pour les modèles. La commande `df.head()` permet de revoir les premières lignes, tandis que `df.shape` indique le nombre de lignes et de colonnes.

## 3. Vérification des valeurs manquantes

In [4]:
missing_values = df.isnull().sum()
missing_values

Age                         0
Attrition                   0
BusinessTravel              0
DailyRate                   0
Department                  0
DistanceFromHome            0
Education                   0
EducationField              0
EmployeeCount               0
EmployeeNumber              0
EnvironmentSatisfaction     0
Gender                      0
HourlyRate                  0
JobInvolvement              0
JobLevel                    0
JobRole                     0
JobSatisfaction             0
MaritalStatus               0
MonthlyIncome               0
MonthlyRate                 0
NumCompaniesWorked          0
Over18                      0
OverTime                    0
PercentSalaryHike           0
PerformanceRating           0
RelationshipSatisfaction    0
StandardHours               0
StockOptionLevel            0
TotalWorkingYears           0
TrainingTimesLastYear       0
WorkLifeBalance             0
YearsAtCompany              0
YearsInCurrentRole          0
YearsSince

**Interprétation :** Cette étape permet de vérifier si certaines colonnes contiennent des valeurs manquantes. Si toutes les valeurs sont égales à 0, cela signifie qu'il n'y a pas de valeurs manquantes apparentes dans le dataset.

## 4. Vérification des doublons

In [5]:
duplicate_count = df.duplicated().sum()
duplicate_count

np.int64(0)

**Interprétation :** Les doublons peuvent influencer l'apprentissage en donnant trop d'importance à certaines observations. Cette vérification permet de savoir si le dataset contient des lignes répétées avant la suite du prétraitement.

## 5. Suppression des colonnes non utiles

In [6]:
columns_to_drop = ["EmployeeCount", "EmployeeNumber", "Over18", "StandardHours"]

# On supprime seulement les colonnes qui existent dans le dataset.
existing_columns_to_drop = [col for col in columns_to_drop if col in df.columns]
df = df.drop(columns=existing_columns_to_drop)

existing_columns_to_drop

['EmployeeCount', 'EmployeeNumber', 'Over18', 'StandardHours']

In [7]:
df.shape

(1470, 31)

**Interprétation :** Les colonnes supprimées sont considérées comme peu utiles pour l'apprentissage. Certaines sont constantes, comme `EmployeeCount`, `Over18` ou `StandardHours`, et `EmployeeNumber` correspond à un identifiant d'employé. Ces informations n'apportent pas réellement de signal général pour prédire l'attrition.

## 6. Séparation de la variable cible et des variables explicatives

In [8]:
X = df.drop("Attrition", axis=1)
y = df["Attrition"].map({"No": 0, "Yes": 1})

print("Dimensions de X :", X.shape)
print("Dimensions de y :", y.shape)
print("Valeurs de la cible après transformation :")
print(y.value_counts())

Dimensions de X : (1470, 30)
Dimensions de y : (1470,)
Valeurs de la cible après transformation :
Attrition
0    1233
1     237
Name: count, dtype: int64


**Interprétation :** `Attrition` est la variable cible à prédire. Elle est transformée en variable numérique pour être compatible avec les futurs modèles : **Yes** devient `1`, ce qui signifie que l'employé quitte l'entreprise, et **No** devient `0`, ce qui signifie qu'il reste.

## 7. Identification des variables numériques et catégorielles

In [9]:
numeric_features = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()

print(f"Nombre de variables numériques : {len(numeric_features)}")
print(numeric_features)

print(f"\nNombre de variables catégorielles : {len(categorical_features)}")
print(categorical_features)

Nombre de variables numériques : 23
['Age', 'DailyRate', 'DistanceFromHome', 'Education', 'EnvironmentSatisfaction', 'HourlyRate', 'JobInvolvement', 'JobLevel', 'JobSatisfaction', 'MonthlyIncome', 'MonthlyRate', 'NumCompaniesWorked', 'PercentSalaryHike', 'PerformanceRating', 'RelationshipSatisfaction', 'StockOptionLevel', 'TotalWorkingYears', 'TrainingTimesLastYear', 'WorkLifeBalance', 'YearsAtCompany', 'YearsInCurrentRole', 'YearsSinceLastPromotion', 'YearsWithCurrManager']

Nombre de variables catégorielles : 7
['BusinessTravel', 'Department', 'EducationField', 'Gender', 'JobRole', 'MaritalStatus', 'OverTime']


**Interprétation :** Cette étape permet de distinguer les variables numériques des variables catégorielles. Les variables catégorielles, comme le département ou le rôle du poste, doivent être encodées avant l'entraînement des modèles, car les algorithmes utilisent des valeurs numériques.

## 8. Préprocesseur pour Decision Tree et Random Forest

In [10]:
preprocessor_tree_rf = ColumnTransformer(
    transformers=[
        ("numeric", "passthrough", numeric_features),
        ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

preprocessor_tree_rf

ColumnTransformer(transformers=[('numeric', 'passthrough',
                                 ['Age', 'DailyRate', 'DistanceFromHome',
                                  'Education', 'EnvironmentSatisfaction',
                                  'HourlyRate', 'JobInvolvement', 'JobLevel',
                                  'JobSatisfaction', 'MonthlyIncome',
                                  'MonthlyRate', 'NumCompaniesWorked',
                                  'PercentSalaryHike', 'PerformanceRating',
                                  'RelationshipSatisfaction',
                                  'StockOptionLevel', 'TotalWorkingYears',
                                  'TrainingTimesLastYear', 'WorkLifeBalance',
                                  'YearsAtCompany', 'YearsInCurrentRole',
                                  'YearsSinceLastPromotion',
                                  'YearsWithCurrManager']),
                                ('categorical',
                                 OneHotEncoder(handle_unknown='ignore'),
                                 ['BusinessTravel', 'Department',
                                  'EducationField', 'Gender', 'JobRole',
                                  'MaritalStatus', 'OverTime'])])

**Interprétation :** Pour les modèles comme **Decision Tree** et **Random Forest**, les variables numériques peuvent être conservées telles quelles. Ces modèles ne nécessitent pas forcément de scaling. En revanche, les variables catégorielles doivent être transformées avec `OneHotEncoder` pour devenir numériques.

## 9. Préprocesseur pour SVM

In [11]:
preprocessor_svm = ColumnTransformer(
    transformers=[
        ("numeric", StandardScaler(), numeric_features),
        ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

preprocessor_svm

ColumnTransformer(transformers=[('numeric', StandardScaler(),
                                 ['Age', 'DailyRate', 'DistanceFromHome',
                                  'Education', 'EnvironmentSatisfaction',
                                  'HourlyRate', 'JobInvolvement', 'JobLevel',
                                  'JobSatisfaction', 'MonthlyIncome',
                                  'MonthlyRate', 'NumCompaniesWorked',
                                  'PercentSalaryHike', 'PerformanceRating',
                                  'RelationshipSatisfaction',
                                  'StockOptionLevel', 'TotalWorkingYears',
                                  'TrainingTimesLastYear', 'WorkLifeBalance',
                                  'YearsAtCompany', 'YearsInCurrentRole',
                                  'YearsSinceLastPromotion',
                                  'YearsWithCurrManager']),
                                ('categorical',
                                 OneHotEncoder(handle_unknown='ignore'),
                                 ['BusinessTravel', 'Department',
                                  'EducationField', 'Gender', 'JobRole',
                                  'MaritalStatus', 'OverTime'])])

**Interprétation :** Le modèle **SVM** est sensible à l'échelle des variables. C'est pourquoi les variables numériques seront standardisées avec `StandardScaler`. Les variables catégorielles seront aussi encodées avec `OneHotEncoder`.

## 10. Train/Test split

In [12]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train :", X_train.shape)
print("X_test :", X_test.shape)
print("y_train :", y_train.shape)
print("y_test :", y_test.shape)

X_train : (1176, 30)
X_test : (294, 30)
y_train : (1176,)
y_test : (294,)


In [13]:
print("Distribution de y_train :")
print(y_train.value_counts(normalize=True) * 100)

print("\nDistribution de y_test :")
print(y_test.value_counts(normalize=True) * 100)

Distribution de y_train :
Attrition
0    83.843537
1    16.156463
Name: proportion, dtype: float64

Distribution de y_test :
Attrition
0    84.013605
1    15.986395
Name: proportion, dtype: float64


**Interprétation :** Le dataset est divisé en données d'entraînement et données de test. L'argument `stratify=y` permet de garder une proportion similaire de la variable **Attrition** dans les deux ensembles. C'est important car la classe `1` correspond aux employés qui quittent l'entreprise et elle est minoritaire.

## 11. Sauvegarde des données préparées

In [14]:
processed_dir = "../data/processed"

# Le dossier est créé seulement s'il n'existe pas déjà.
import os
os.makedirs(processed_dir, exist_ok=True)

X_train.to_csv(f"{processed_dir}/X_train.csv", index=False)
X_test.to_csv(f"{processed_dir}/X_test.csv", index=False)
y_train.to_frame(name="Attrition").to_csv(f"{processed_dir}/y_train.csv", index=False)
y_test.to_frame(name="Attrition").to_csv(f"{processed_dir}/y_test.csv", index=False)

print("Fichiers sauvegardés dans ../data/processed")

Fichiers sauvegardés dans ../data/processed


**Interprétation :** Les ensembles `X_train`, `X_test`, `y_train` et `y_test` sont sauvegardés dans le dossier `data/processed`. Cela permettra de les réutiliser plus facilement dans les notebooks suivants, notamment pour l'entraînement des modèles.

## Conclusion partielle

Dans ce notebook, les données ont été préparées pour la suite du projet. Les colonnes non utiles ont été supprimées, car elles sont constantes, identifiantes ou peu informatives pour l'apprentissage.

La variable cible **Attrition** a été transformée en valeurs numériques : `Yes = 1` et `No = 0`. Les variables numériques et catégorielles ont ensuite été identifiées.

L'encodage des variables catégorielles sera nécessaire pour les modèles, car les algorithmes travaillent avec des données numériques. Le scaling sera surtout nécessaire pour **SVM**, car ce modèle est sensible à l'échelle des variables.

Les données ont aussi été divisées en ensembles d'entraînement et de test avec une stratification sur la cible, afin de conserver la même proportion d'employés qui quittent ou restent dans les deux ensembles.

La prochaine étape sera l'entraînement du modèle **Decision Tree**.